In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold

# 1. Load Dataset

In [2]:
# Load train and test data 
BASE_DIR = Path("../data")
train = pd.read_csv(BASE_DIR / "train.csv")
test  = pd.read_csv(BASE_DIR / "test.csv")

# Re-parse kolom "Date"
train["Date"] = pd.to_datetime(train["Date"])
test["Date"]  = pd.to_datetime(test["Date"])

print(f"\ntrain: {train.shape}")
print(f"test : {test.shape}")
train.head()


train: (1866532, 10)
test : (465999, 10)


,Unnamed: 0,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,Target
0,0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,0.000730
1,1,20170104_7412,2017-01-04,7412,719.0,725.0,719.0,721.0,201400,0.000000
2,2,20170104_7408,2017-01-04,7408,2459.0,2518.0,2447.0,2500.0,110900,0.004421
3,3,20170104_7315,2017-01-04,7315,465.0,494.0,465.0,493.0,41100,-0.004032
4,4,20170104_7313,2017-01-04,7313,3055.0,3150.0,3045.0,3135.0,248600,-0.009693


In [3]:
#split data into train and validation sets
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(train)):
    print(f"Fold {fold}: train={len(train_idx)} rows, val={len(val_idx)} rows")

# ambil fold terakhir sebagai train/validation final
train_idx, val_idx = list(tscv.split(train))[-1]

train_sub = train.iloc[train_idx].reset_index(drop=True)
val_sub   = train.iloc[val_idx].reset_index(drop=True)

print(f"\nTrain final : {train_sub.shape}")
print(f"Validation  : {val_sub.shape}")

Fold 0: train=311092 rows, val=311088 rows
Fold 1: train=622180 rows, val=311088 rows
Fold 2: train=933268 rows, val=311088 rows
Fold 3: train=1244356 rows, val=311088 rows
Fold 4: train=1555444 rows, val=311088 rows

Train final : (1555444, 10)
Validation  : (311088, 10)


# Preprocessing

In [4]:
# drop missing values
train_sub = train_sub.dropna().reset_index(drop=True)
val_sub   = val_sub.dropna().reset_index(drop=True)
test      = test.dropna().reset_index(drop=True)

# Feature Engineering

In [5]:
train_sub = train_sub.copy()
val_sub   = val_sub.copy()
test      = test.copy()

train_sub["_split"] = "train_sub"
val_sub["_split"]   = "val_sub"
test["_split"]      = "test"
full = pd.concat([train_sub, val_sub, test], axis=0, ignore_index=True)
full = full.sort_values(["SecuritiesCode", "Date"]).reset_index(drop=True)

g = full.groupby("SecuritiesCode", sort=False)

# 1. Fitur Temporal
full["year"]       = full["Date"].dt.year
full["month"]      = full["Date"].dt.month
full["day"]        = full["Date"].dt.day
full["dayofweek"]  = full["Date"].dt.dayofweek
full["quarter"]    = full["Date"].dt.quarter
full["dayofyear"]  = full["Date"].dt.dayofyear
full["weekofyear"] = full["Date"].dt.isocalendar().week.astype(int)

# 2. Fitur Return (N-day)
full["ret_1d"]  = g["Close"].pct_change()
full["ret_5d"]  = g["Close"].pct_change(5)
full["ret_20d"] = g["Close"].pct_change(20)

# 3. Fitur Lag (explicit shift)
full["ret_1d_lag1"] = g["ret_1d"].shift(1)
full["ret_1d_lag2"] = g["ret_1d"].shift(2)
full["ret_1d_lag3"] = g["ret_1d"].shift(3)

# 4. Fitur Rolling Window (agregasi statistik)
full["vol_5d"]  = g["ret_1d"].transform(lambda s: s.rolling(5,  min_periods=2).std())
full["vol_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20, min_periods=5).std())

vol_ma20             = g["Volume"].transform(lambda s: s.rolling(20, min_periods=1).mean())
full["volume_ratio"] = full["Volume"] / vol_ma20
del vol_ma20

# 5. Fitur Bentuk Intraday
full["intraday_range"] = (full["High"] - full["Low"]) / full["Close"]
full["close_open"]     = (full["Close"] - full["Open"]) / full["Open"]
full["high_close"]     = (full["High"] - full["Close"]) / full["Close"]
full["low_close"]      = (full["Close"] - full["Low"]) / full["Close"]

# 6. Fitur Volume
full["log_volume"] = np.log1p(full["Volume"])

# 7. Fitur Cross-sectional Rank
full["ret_rank"]    = full.groupby("Date")["ret_1d"].rank(pct=True)
full["volume_rank"] = full.groupby("Date")["Volume"].rank(pct=True)

# pisahkan kembali
train_sub = full[full["_split"] == "train_sub"].drop(columns="_split").reset_index(drop=True)
val_sub   = full[full["_split"] == "val_sub"].drop(columns="_split").reset_index(drop=True)
test      = full[full["_split"] == "test"].drop(columns="_split").reset_index(drop=True)

print("Fitur ditambahkan: Temporal, Return, Lag, Rolling Window, Intraday, Volume, Rank.")
print(f"train_sub: {train_sub.shape}, val_sub: {val_sub.shape}, test: {test.shape}")

Fitur ditambahkan: Temporal, Return, Lag, Rolling Window, Intraday, Volume, Rank.
train_sub: (1551801, 33), val_sub: (308228, 33), test: (464894, 33)


In [6]:
#outlier handling :capping
from math import isnan

def handle_outliers(df, column, method="iqr", action="cap", bounds=None):
    if column not in df.columns:
        raise KeyError(f"Column tidak ditemukan: {column}")
    if not pd.api.types.is_numeric_dtype(df[column]):
        raise TypeError(f"Column harus numerik: {column}")
    if method not in {"iqr", "zscore"}:
        raise ValueError("method harus 'iqr' atau 'zscore'")
    if action not in {"remove", "cap"}:
        raise ValueError("action harus 'remove' atau 'cap'")

    result = df.copy()
    values = result[column]

    if bounds is None:
        if method == "iqr":
            q1, q3 = values.quantile([0.25, 0.75])
            spread = q3 - q1
            lower = q1 - 1.5 * spread
            upper = q3 + 1.5 * spread
        else:
            mean = values.mean()
            std = values.std()
            lower = mean - 3 * std
            upper = mean + 3 * std
    else:
        lower, upper = bounds

    outlier_mask = values.notna() & ((values < lower) | (values > upper))
    before_rows = len(result)
    outlier_count = int(outlier_mask.sum())

    if action == "remove":
        result = result.loc[~outlier_mask].reset_index(drop=True)
        changed = outlier_count
        print(f"[{column}] {method} removal: {outlier_count} baris dihapus")
    else:
        result[column] = values.clip(lower=lower, upper=upper)
        changed = int((result[column] != values).fillna(False).sum())
        print(f"[{column}] {method} capping: {changed} nilai dibatasi")

    print(f"  Sebelum: {before_rows:,} baris | Sesudah: {len(result):,} baris")
    print(f"  Batas: [{lower:.6g}, {upper:.6g}] | Outlier terdeteksi: {outlier_count:,}")
    return result, (lower, upper)


# --- Terapkan ke SEMUA kolom skewed: train_sub -> hitung bounds, val_sub & test -> pakai bounds train_sub ---
CAP_COLS = [
    "ret_1d", "ret_5d", "ret_20d",
    "ret_1d_lag1", "ret_1d_lag2", "ret_1d_lag3",
    "vol_5d", "vol_20d",
    "volume_ratio",
]

cap_bounds = {}
for col in CAP_COLS:
    train_sub, bounds = handle_outliers(train_sub, column=col, method="iqr", action="cap")
    cap_bounds[col] = bounds

for col in CAP_COLS:
    val_sub, _ = handle_outliers(val_sub, column=col, method="iqr", action="cap", bounds=cap_bounds[col])
    test, _    = handle_outliers(test, column=col, method="iqr", action="cap", bounds=cap_bounds[col])

print("\nSemua kolom skewed sudah di-cap (bounds dari train_sub):")
for col, (lo, hi) in cap_bounds.items():
    print(f"  {col:15s}: [{lo:.6g}, {hi:.6g}]")

[ret_1d] iqr capping: 109350 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403926, 0.0402078] | Outlier terdeteksi: 107,376
[ret_5d] iqr capping: 117121 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0963203, 0.0970418] | Outlier terdeteksi: 107,251
[ret_20d] iqr capping: 130836 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.208932, 0.211786] | Outlier terdeteksi: 91,356
[ret_1d_lag1] iqr capping: 111129 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403739, 0.0401712] | Outlier terdeteksi: 107,181
[ret_1d_lag2] iqr capping: 112987 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403688, 0.0401496] | Outlier terdeteksi: 107,065
[ret_1d_lag3] iqr capping: 114797 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403737, 0.0401183] | Outlier terdeteksi: 106,901
[vol_5d] iqr capping:

In [7]:
#security code target encoding
mean_target = train_sub.groupby("SecuritiesCode")["Target"].mean()
global_mean = train_sub["Target"].mean()

train_sub["SecuritiesCode_te"] = train_sub["SecuritiesCode"].map(mean_target)
val_sub["SecuritiesCode_te"]   = val_sub["SecuritiesCode"].map(mean_target).fillna(global_mean)
test["SecuritiesCode_te"]      = test["SecuritiesCode"].map(mean_target).fillna(global_mean)

print(f"SecuritiesCode_te dibuat. mean (train_sub)={train_sub['SecuritiesCode_te'].mean():.6f}")
print(f"SecuritiesCode_te dibuat. mean (val_sub)={val_sub['SecuritiesCode_te'].mean():.6f}")
print(f"SecuritiesCode_te dibuat. mean (test)={test['SecuritiesCode_te'].mean():.6f}")

SecuritiesCode_te dibuat. mean (train_sub)=0.000204
SecuritiesCode_te dibuat. mean (val_sub)=0.000267
SecuritiesCode_te dibuat. mean (test)=0.000266


In [8]:
# Finalisasi dataset untuk train_sub, val_sub, dan test
# Kolom OHLCV tidak digunakan sebagai fitur model setelah feature engineering.
feature_drop = ["Open", "High", "Low", "Close", "Volume"]
identifier_drop = ["Unnamed: 0", "RowId", "SecuritiesCode", "Date"]

train_final = train_sub.drop(columns=feature_drop, errors="ignore").dropna().reset_index(drop=True)
val_final   = val_sub.drop(columns=feature_drop, errors="ignore").dropna().reset_index(drop=True)
test_final  = test.drop(columns=feature_drop, errors="ignore").dropna().reset_index(drop=True)

train_X = train_final.drop(columns="Target", errors="ignore")
train_y = train_final["Target"]
val_X   = val_final.drop(columns="Target", errors="ignore")
val_y   = val_final["Target"]
test_X  = test_final.drop(columns="Target", errors="ignore")
test_y  = test_final["Target"]                      

train_X = train_X.drop(columns=identifier_drop, errors="ignore")
val_X   = val_X.drop(columns=identifier_drop, errors="ignore")
test_X  = test_X.drop(columns=identifier_drop, errors="ignore")

print(f"Fitur final ({len(train_X.columns)} kolom): {list(train_X.columns)}")
print(f"train_sub: {train_X.shape}, target: {train_y.shape}")
print(f"val_sub  : {val_X.shape}, target: {val_y.shape}")
print(f"test     : {test_X.shape}, target: {test_y.shape}")   

Fitur final (24 kolom): ['year', 'month', 'day', 'dayofweek', 'quarter', 'dayofyear', 'weekofyear', 'ret_1d', 'ret_5d', 'ret_20d', 'ret_1d_lag1', 'ret_1d_lag2', 'ret_1d_lag3', 'vol_5d', 'vol_20d', 'volume_ratio', 'intraday_range', 'close_open', 'high_close', 'low_close', 'log_volume', 'ret_rank', 'volume_rank', 'SecuritiesCode_te']
train_sub: (1512321, 24), target: (1512321,)
val_sub  : (307872, 24), target: (307872,)
test     : (464730, 24), target: (464730,)


In [9]:
# Feature scaling
from sklearn.preprocessing import StandardScaler, MinMaxScaler

FEATURE_COLS = train_X.columns.tolist()
TARGET_COL = "Target"

X_train = train_X[FEATURE_COLS].to_numpy(dtype=np.float64)
y_train = train_y.to_numpy(dtype=np.float64).reshape(-1, 1)
X_val   = val_X[FEATURE_COLS].to_numpy(dtype=np.float64)
y_val   = val_y.to_numpy(dtype=np.float64).reshape(-1, 1)
X_test  = test_X[FEATURE_COLS].to_numpy(dtype=np.float64)
y_test  = test_y.to_numpy(dtype=np.float64).reshape(-1, 1)      

# Pipeline Machine Learning
scaler_X_svr = StandardScaler()
scaler_y_svr = StandardScaler()
X_train_svr = scaler_X_svr.fit_transform(X_train)
X_val_svr   = scaler_X_svr.transform(X_val)
X_test_svr  = scaler_X_svr.transform(X_test)
y_train_svr = scaler_y_svr.fit_transform(y_train).ravel()
y_val_svr   = scaler_y_svr.transform(y_val).ravel()
y_test_svr  = scaler_y_svr.transform(y_test).ravel()             

# Pipeline Deep Learning
scaler_X_lstm = MinMaxScaler()
scaler_y_lstm = MinMaxScaler()
X_train_lstm = scaler_X_lstm.fit_transform(X_train)
X_val_lstm   = scaler_X_lstm.transform(X_val)
X_test_lstm  = scaler_X_lstm.transform(X_test)
y_train_lstm = scaler_y_lstm.fit_transform(y_train).ravel()
y_val_lstm   = scaler_y_lstm.transform(y_val).ravel()
y_test_lstm  = scaler_y_lstm.transform(y_test).ravel()           

print(f"Jumlah fitur: {len(FEATURE_COLS)}")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

Jumlah fitur: 24
X_train: (1512321, 24) | X_val: (307872, 24) | X_test: (464730, 24)


# Modelling

In [10]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# Build LSTM model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_lstm.shape[1]

X_train_lstm_3d = X_train_lstm.reshape(X_train_lstm.shape[0], 1, n_features)
X_val_lstm_3d   = X_val_lstm.reshape(X_val_lstm.shape[0], 1, n_features)
X_test_lstm_3d  = X_test_lstm.reshape(X_test_lstm.shape[0], 1, n_features)

lstm_model = Sequential([
    Input(shape=(1, n_features)),
    LSTM(50, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(optimizer="adam", loss="mean_squared_error")
lstm_model.summary()

print(f"X_train_lstm: {X_train_lstm_3d.shape}")
print(f"X_val_lstm  : {X_val_lstm_3d.shape}")
print(f"X_test_lstm : {X_test_lstm_3d.shape}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        15,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,051 (58.79 KB)

 Trainable params: 15,051 (58.79 KB)

 Non-trainable params: 0 (0.00 B)

X_train_lstm: (1512321, 1, 24)
X_val_lstm  : (307872, 1, 24)
X_test_lstm : (464730, 1, 24)


In [12]:
# training model
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training LSTM pada {len(X_train_lstm_3d):,} sample train "
      f"dan {len(X_val_lstm_3d):,} sample validation (maks 100 epoch) ...")

history = lstm_model.fit(
    X_train_lstm_3d, y_train_lstm,              
    validation_data=(X_val_lstm_3d, y_val_lstm),
    batch_size=16,
    epochs=100,
    callbacks=[early_stop],
    verbose=1,
    shuffle=False,
)

Training LSTM pada 1,512,321 sample train dan 307,872 sample validation (maks 100 epoch) ...
Epoch 1/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 451s 5ms/step - loss: 4.9248e-04 - val_loss: 4.9557e-04
Epoch 2/100
39959/94521 ━━━━━━━━━━━━━━━━━━━━ 4:37 5ms/step - loss: 3.6236e-04

KeyboardInterrupt: 

In [ ]:
# evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Generate predictions on the test set in the scaled target space.
y_pred_lstm_scaled = lstm_model.predict(X_test_lstm_3d, verbose=0).ravel()
y_true_lstm_scaled = y_test_lstm.ravel()

# Inverse transform back to the original target scale for meaningful metrics.
y_pred_lstm = scaler_y_lstm.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1)).ravel()
y_true_lstm = scaler_y_lstm.inverse_transform(y_true_lstm_scaled.reshape(-1, 1)).ravel()

mse_lstm = mean_squared_error(y_true_lstm, y_pred_lstm)
rmse_lstm = np.sqrt(mse_lstm)
mae_lstm = mean_absolute_error(y_true_lstm, y_pred_lstm)

print(f"LSTM -> MSE: {mse_lstm:.8f} | RMSE: {rmse_lstm:.8f} | MAE: {mae_lstm:.8f}")

### hyperparam

In [ ]:
%pip install optuna

In [ ]:
# Hyperparameter Tuning LSTM Pertama dengan Optuna
import optuna
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_lstm.shape[1]

def build_lstm(units, dropout_rate, learning_rate, n_features):
    model = Sequential([
        Input(shape=(1, n_features)),
        LSTM(units, activation="relu"),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="mean_squared_error")
    return model


def objective(trial):
    units         = trial.suggest_categorical("units", [32, 50, 64, 100, 128])
    dropout_rate  = trial.suggest_float("dropout_rate", 0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])

    model = build_lstm(units, dropout_rate, learning_rate, n_features)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=0,
    )

    history = model.fit(
        X_train_lstm_3d, y_train_lstm,
        validation_data=(X_val_lstm_3d, y_val_lstm),
        batch_size=batch_size,
        epochs=50,              # dikurangi supaya tuning tidak terlalu lama
        callbacks=[early_stop],
        verbose=0,
        shuffle=False,
    )

    best_val_loss = min(history.history["val_loss"])

    # simpan info tambahan di trial (opsional, bisa dilihat lagi nanti)
    trial.set_user_attr("epochs_trained", len(history.history["val_loss"]))

    return best_val_loss


study = optuna.create_study(direction="minimize", study_name="lstm_tuning")
study.optimize(objective, n_trials=20, show_progress_bar=True)

print("\n=== Hasil Tuning Optuna ===")
print(f"Best val_loss (MSE, scaled): {study.best_value:.8f}")
print(f"Best params: {study.best_params}")

In [ ]:
trials_df = study.trials_dataframe().sort_values("value").reset_index(drop=True)
print(trials_df[["number", "value", "params_units", "params_dropout_rate",
                  "params_learning_rate", "params_batch_size"]])

In [ ]:
best_params = study.best_params

final_lstm = build_lstm(
    units=best_params["units"],
    dropout_rate=best_params["dropout_rate"],
    learning_rate=best_params["learning_rate"],
    n_features=n_features,
)

early_stop_final = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

history_final = final_lstm.fit(
    X_train_lstm_3d, y_train_lstm,
    validation_data=(X_val_lstm_3d, y_val_lstm),
    batch_size=best_params["batch_size"],
    epochs=100,
    callbacks=[early_stop_final],
    verbose=1,
    shuffle=False,
)

# Evaluasi di test set
y_pred_tuned_scaled = final_lstm.predict(X_test_lstm_3d, verbose=0).ravel()
y_true_tuned_scaled = y_test_lstm.ravel()

y_pred_tuned = scaler_y_lstm.inverse_transform(y_pred_tuned_scaled.reshape(-1, 1)).ravel()
y_true_tuned = scaler_y_lstm.inverse_transform(y_true_tuned_scaled.reshape(-1, 1)).ravel()

mse_tuned  = mean_squared_error(y_true_tuned, y_pred_tuned)
rmse_tuned = np.sqrt(mse_tuned)
mae_tuned  = mean_absolute_error(y_true_tuned, y_pred_tuned)

print(f"LSTM (tuned Optuna) -> MSE: {mse_tuned:.8f} | RMSE: {rmse_tuned:.8f} | MAE: {mae_tuned:.8f}")

In [ ]:
import optuna.visualization.matplotlib as ov_mpl
import matplotlib.pyplot as plt

ov_mpl.plot_optimization_history(study)
plt.show()

ov_mpl.plot_param_importances(study)
plt.show()

# MODELLING SMUA JENIS

In [ ]:
# LSTM model
# Build LSTM model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_lstm.shape[1]

X_train_lstm_3d = X_train_lstm.reshape(X_train_lstm.shape[0], 1, n_features)
X_val_lstm_3d   = X_val_lstm.reshape(X_val_lstm.shape[0], 1, n_features)
X_test_lstm_3d  = X_test_lstm.reshape(X_test_lstm.shape[0], 1, n_features)

lstm_model2 = Sequential([
    Input(shape=(1, n_features)),
    LSTM(64, activation="relu", return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

lstm_model2.compile(optimizer="adam", loss="mean_squared_error")
lstm_model2.summary()

print(f"X_train_lstm: {X_train_lstm_3d.shape}")
print(f"X_val_lstm  : {X_val_lstm_3d.shape}")
print(f"X_test_lstm : {X_test_lstm_3d.shape}")

# training model
early_stop2 = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training LSTM2 pada {len(X_train_lstm_3d):,} sample train "
      f"dan {len(X_val_lstm_3d):,} sample validation (maks 100 epoch) ...")

history2 = lstm_model2.fit(
    X_train_lstm_3d, y_train_lstm,
    validation_data=(X_val_lstm_3d, y_val_lstm),
    batch_size=64,
    epochs=100,
    callbacks=[early_stop2],
    verbose=1,
    shuffle=False,
)

# predict
y_pred_lstm2_scaled = lstm_model2.predict(X_test_lstm_3d, verbose=0).ravel()
y_true_lstm2_scaled = y_test_lstm.ravel()

# evaluation metrics
y_pred_lstm2 = scaler_y_lstm.inverse_transform(y_pred_lstm2_scaled.reshape(-1, 1)).ravel()
y_true_lstm2 = scaler_y_lstm.inverse_transform(y_true_lstm2_scaled.reshape(-1, 1)).ravel()

mse_lstm2  = mean_squared_error(y_true_lstm2, y_pred_lstm2)
rmse_lstm2 = np.sqrt(mse_lstm2)
mae_lstm2  = mean_absolute_error(y_true_lstm2, y_pred_lstm2)

print(f"LSTM2 -> MSE: {mse_lstm2:.8f} | RMSE: {rmse_lstm2:.8f} | MAE: {mae_lstm2:.8f}")

In [ ]:
#MLP
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_lstm.shape[1]

mlp_model = Sequential([
    Input(shape=(n_features,)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

mlp_model.compile(optimizer="adam", loss="mean_squared_error")
mlp_model.summary()

print(f"X_train_lstm: {X_train_lstm.shape}")
print(f"X_val_lstm  : {X_val_lstm.shape}")
print(f"X_test_lstm : {X_test_lstm.shape}")

# training model
early_stop_mlp = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training MLP pada {len(X_train_lstm):,} sample train "
      f"dan {len(X_val_lstm):,} sample validation (maks 100 epoch) ...")

history_mlp = mlp_model.fit(
    X_train_lstm, y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    batch_size=64,
    epochs=100,
    callbacks=[early_stop_mlp],
    verbose=1,
    shuffle=False,
)

# predict
y_pred_mlp_scaled = mlp_model.predict(X_test_lstm, verbose=0).ravel()
y_true_mlp_scaled = y_test_lstm.ravel()

# evaluation metrics
y_pred_mlp = scaler_y_lstm.inverse_transform(y_pred_mlp_scaled.reshape(-1, 1)).ravel()
y_true_mlp = scaler_y_lstm.inverse_transform(y_true_mlp_scaled.reshape(-1, 1)).ravel()

mse_mlp  = mean_squared_error(y_true_mlp, y_pred_mlp)
rmse_mlp = np.sqrt(mse_mlp)
mae_mlp  = mean_absolute_error(y_true_mlp, y_pred_mlp)

print(f"MLP -> MSE: {mse_mlp:.8f} | RMSE: {rmse_mlp:.8f} | MAE: {mae_mlp:.8f}")

In [ ]:
#GRU
from tensorflow.keras.layers import GRU

n_features = X_train_lstm.shape[1]

X_train_gru_3d = X_train_lstm.reshape(X_train_lstm.shape[0], 1, n_features)
X_val_gru_3d   = X_val_lstm.reshape(X_val_lstm.shape[0], 1, n_features)
X_test_gru_3d  = X_test_lstm.reshape(X_test_lstm.shape[0], 1, n_features)

gru_model = Sequential([
    Input(shape=(1, n_features)),
    GRU(64, activation="relu", return_sequences=True),
    Dropout(0.2),
    GRU(32, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

gru_model.compile(optimizer="adam", loss="mean_squared_error")
gru_model.summary()

print(f"X_train_gru: {X_train_gru_3d.shape}")
print(f"X_val_gru  : {X_val_gru_3d.shape}")
print(f"X_test_gru : {X_test_gru_3d.shape}")

# training model
early_stop_gru = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training GRU pada {len(X_train_gru_3d):,} sample train "
      f"dan {len(X_val_gru_3d):,} sample validation (maks 100 epoch) ...")

history_gru = gru_model.fit(
    X_train_gru_3d, y_train_lstm,
    validation_data=(X_val_gru_3d, y_val_lstm),
    batch_size=64,
    epochs=100,
    callbacks=[early_stop_gru],
    verbose=1,
    shuffle=False,
)

# predict
y_pred_gru_scaled = gru_model.predict(X_test_gru_3d, verbose=0).ravel()
y_true_gru_scaled = y_test_lstm.ravel()

# evaluation metrics
y_pred_gru = scaler_y_lstm.inverse_transform(y_pred_gru_scaled.reshape(-1, 1)).ravel()
y_true_gru = scaler_y_lstm.inverse_transform(y_true_gru_scaled.reshape(-1, 1)).ravel()

mse_gru  = mean_squared_error(y_true_gru, y_pred_gru)
rmse_gru = np.sqrt(mse_gru)
mae_gru  = mean_absolute_error(y_true_gru, y_pred_gru)

print(f"GRU -> MSE: {mse_gru:.8f} | RMSE: {rmse_gru:.8f} | MAE: {mae_gru:.8f}")

In [ ]:
# CNN 1D
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D

n_features = X_train_lstm.shape[1]

X_train_cnn_3d = X_train_lstm.reshape(X_train_lstm.shape[0], 1, n_features)
X_val_cnn_3d   = X_val_lstm.reshape(X_val_lstm.shape[0], 1, n_features)
X_test_cnn_3d  = X_test_lstm.reshape(X_test_lstm.shape[0], 1, n_features)

cnn_model = Sequential([
    Input(shape=(1, n_features)),
    Conv1D(64, kernel_size=1, activation="relu", padding="causal"),
    Conv1D(32, kernel_size=1, activation="relu", padding="causal"),
    GlobalAveragePooling1D(),
    Dropout(0.2),
    Dense(1)
])

cnn_model.compile(optimizer="adam", loss="mean_squared_error")
cnn_model.summary()

print(f"X_train_cnn: {X_train_cnn_3d.shape}")
print(f"X_val_cnn  : {X_val_cnn_3d.shape}")
print(f"X_test_cnn : {X_test_cnn_3d.shape}")

# training model
early_stop_cnn = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training CNN1D pada {len(X_train_cnn_3d):,} sample train "
      f"dan {len(X_val_cnn_3d):,} sample validation (maks 100 epoch) ...")

history_cnn = cnn_model.fit(
    X_train_cnn_3d, y_train_lstm,
    validation_data=(X_val_cnn_3d, y_val_lstm),
    batch_size=64,
    epochs=100,
    callbacks=[early_stop_cnn],
    verbose=1,
    shuffle=False,
)

# predict
y_pred_cnn_scaled = cnn_model.predict(X_test_cnn_3d, verbose=0).ravel()
y_true_cnn_scaled = y_test_lstm.ravel()

# evaluation metrics
y_pred_cnn = scaler_y_lstm.inverse_transform(y_pred_cnn_scaled.reshape(-1, 1)).ravel()
y_true_cnn = scaler_y_lstm.inverse_transform(y_true_cnn_scaled.reshape(-1, 1)).ravel()

mse_cnn  = mean_squared_error(y_true_cnn, y_pred_cnn)
rmse_cnn = np.sqrt(mse_cnn)
mae_cnn  = mean_absolute_error(y_true_cnn, y_pred_cnn)

print(f"CNN1D -> MSE: {mse_cnn:.8f} | RMSE: {rmse_cnn:.8f} | MAE: {mae_cnn:.8f}")